# Mini-projeto 1 - Fase 1: MLP para classificação do CIFAR-10

Este notebook orquestra os experimentos da Fase 1 (MLP). A lógica reutilizável (modelo, dados, treino, métricas, checkpointing) vive no pacote `mlp_cifar10` em `../src/`, para que o notebook fique focado em **definir experimentos e reportar resultados** — não em implementação.

**Integrantes do grupo:** _preencher aqui (nome de todos)_

O que este notebook cobre (conforme o enunciado do mini-projeto):
- Treino de um MLP no CIFAR-10 com hiperparâmetros configuráveis.
- Métricas por classe (acurácia) e globais (acurácia, precision, recall, f1).
- Comparação entre variações de hiperparâmetros (camadas, neurônios, taxa de aprendizagem, ativação, regularização, otimizador, dropout, função de erro).
- Cada execução de treino é salva automaticamente em `../results/` (pesos + config + métricas + histórico) — ver `../src/mlp_cifar10/checkpointing.py`.

## 0. Setup do ambiente

- **Local**: rode a partir de um ambiente onde o pacote já foi instalado (`pip install -e .` na pasta `fase1-mlp/`).
- **Google Colab**: a célula abaixo clona o repositório e instala o pacote automaticamente.

In [ ]:
#@title Setup (Colab ou local)
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "<preencher com a URL do repositório>"  # ex.: https://github.com/usuario/redes-neurais.git
    REPO_DIR = Path("/content/redes-neurais")
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    %pip install -q -e {REPO_DIR}/miniprojeto/fase1-mlp
    PROJECT_ROOT = REPO_DIR / "miniprojeto" / "fase1-mlp"
else:
    PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> fase1-mlp/
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
#@title Imports
import torch
import matplotlib.pyplot as plt
import pandas as pd

from mlp_cifar10.config import ExperimentConfig
from mlp_cifar10.data import get_dataloaders, CLASSES
from mlp_cifar10.train import fit
from mlp_cifar10.checkpointing import load_all_metadata
from mlp_cifar10.utils import set_seed, get_device

In [ ]:
#@title Device
device = get_device()
print("Usando dispositivo:", device)

## 1. Experimento baseline

Arquitetura equivalente à do notebook de referência: `3072 -> 64 -> 128 -> 64 -> 10`, ReLU, Adam, entropia cruzada.

In [ ]:
baseline_config = ExperimentConfig(
    run_name="baseline_relu_adam",
    hidden_layers=(64, 128, 64),
    activation="relu",
    dropout=0.0,
    batch_norm=False,
    optimizer="adam",
    loss="cross_entropy",
    learning_rate=1e-3,
    weight_decay=0.0,
    batch_size=64,
    num_epochs=100,
    patience=5,
    notes="Baseline equivalente ao notebook de referência da disciplina.",
)

set_seed(baseline_config.seed)
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=baseline_config.batch_size,
    val_fraction=baseline_config.val_fraction,
    seed=baseline_config.seed,
)

In [ ]:
baseline_result = fit(
    config=baseline_config,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    class_names=list(CLASSES),
    results_dir=RESULTS_DIR,
)

In [ ]:
#@title Curvas de treino (loss/acurácia por época)
history_df = pd.DataFrame(baseline_result["history"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].set_title("Loss")

axes[1].plot(history_df["epoch"], history_df["val_accuracy"], label="val_accuracy", color="green")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Acurácia"); axes[1].legend(); axes[1].set_title("Acurácia de validação")
plt.tight_layout()
plt.show()

In [ ]:
#@title Resultados no teste
print("Métricas globais:")
display(pd.Series(baseline_result["test_scores"]).to_frame("valor"))

print("\nAcurácia por classe:")
display(pd.Series(baseline_result["per_class_accuracy"]).to_frame("acurácia"))

## 2. Busca de hiperparâmetros

Defina abaixo as variações de configuração que você quer comparar com o baseline
(número de camadas/neurônios, taxa de aprendizagem, ativação, regularização,
otimizador, dropout, função de erro, ...). Cada execução é salva automaticamente
em `../results/`, então nada se perde entre uma rodada e outra.

In [ ]:
candidate_configs = [
    baseline_config,
    ExperimentConfig(
        run_name="deeper_dropout",
        hidden_layers=(256, 128, 64, 32),
        activation="relu",
        dropout=0.3,
        optimizer="adam",
        learning_rate=1e-3,
        notes="Rede mais profunda + dropout para reduzir overfitting.",
    ),
    ExperimentConfig(
        run_name="sgd_momentum",
        hidden_layers=(64, 128, 64),
        activation="relu",
        optimizer="sgd",
        momentum=0.9,
        learning_rate=1e-2,
        notes="Trocando o otimizador para SGD com momentum.",
    ),
    # Adicione outras variações conforme os experimentos forem sendo decididos.
]

In [ ]:
experiment_results = {}
for config in candidate_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
    )
    experiment_results[config.run_name] = fit(
        config=config,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        class_names=list(CLASSES),
        results_dir=RESULTS_DIR,
    )

In [ ]:
#@title Tabela comparativa dos experimentos
comparison = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison

## 2.1 Comparando todas as execuções salvas (`results/`)

Como cada `fit(...)` salva automaticamente seus resultados em `results/{model_id}/metadata.json` (skill model-saver), é possível comparar **todas** as execuções já feitas — não só as desta sessão do notebook — lendo direto do disco.

In [ ]:
#@title Comparação de todos os runs salvos em results/
all_runs_df = load_all_metadata(RESULTS_DIR)
cols = [c for c in all_runs_df.columns if c.startswith("metrics.") or c == "model_id"]
all_runs_df[cols].sort_values("metrics.test_accuracy", ascending=False)

## 3. Conclusões

_Preencher com a análise dos resultados: quais mudanças de hiperparâmetro
trouxeram ganho/perda de desempenho e possíveis explicações. Esta seção (e a
tabela comparativa acima) é a base do PPT/relatório a entregar._